In [ ]:
import numpy as np
import plotly.graph_objects as go
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

EVENTS = {
    "DiffiT b192 (256x256)": "/home/david/mnt/ssd_2_sata/python/phd/wc_cv/ml/00005-diffit-256-gpus2-batch192/events.out.tfevents.1774738850.cn-048.1265317.0",
    "DiffiT b256 (256x256)": "/home/david/mnt/ssd_2_sata/python/phd/wc_cv/ml/00006-diffit-256-gpus2-batch256/events.out.tfevents.1774853357.cn-048.1590522.0",
    "DiT-XL (256x256)":      "/home/david/mnt/ssd_2_sata/python/phd/wc_cv/ml/dit-xl2-3cls-20260405_231745/tensorboard/events.out.tfevents.1775420267.cn-043.2986252.0",
    "NiT":                   "/home/david/mnt/ssd_2_sata/python/phd/wc_cv/ml/nit/logs/imagenet_9to4/events.out.tfevents.1775432710.cn-047.174932.0",
}

# Conversion: TB scalar `step` field → images seen.
# IMPORTANT: TB `step` does NOT mean the same thing across these runs!
#  - DiffiT (both runs): writes scalars with `step = kimg` (verified: Train/kimg value ≈ TB step).
#            So images = tb_step * 1000. This is independent of global batch size —
#            the kimg-per-tick is set by kimg_per_tick=4 in the config, not by batch.
#  - DiT-XL: TB step IS the optimizer step. Global batch = 64 * 2 grad-accum * 2 GPU = 256.
#            So images = tb_step * 256.
#  - NiT:    TB step is the optimizer step. dataloader batch = 3 packs of 8192 latent
#            tokens, mixed resolutions {256,512,1024} → ~190 tokens/image avg →
#            ~130 images/step (approximate).
STEP_TO_IMAGES = {
    "DiffiT b192 (256x256)": 1000,  # tb_step is kimg
    "DiffiT b256 (256x256)": 1000,  # tb_step is kimg
    "DiT-XL (256x256)":      256,   # tb_step is optimizer step, global batch 256
    "NiT":                   130,   # approximate
}

COLORS = {
    "DiffiT b192 (256x256)": "#1f77b4",
    "DiffiT b256 (256x256)": "#aec7e8",
    "DiT-XL (256x256)":      "#ff7f0e",
    "NiT":                   "#2ca02c",
    "NiT (256x256)":         "#2ca02c",
    "NiT (512x512)":         "#17becf",
    "NiT (1024x1024)":       "#9467bd",
}

def load(path):
    ea = EventAccumulator(path, size_guidance={"scalars": 0})
    ea.Reload()
    return ea

def scalar(ea, tag):
    evs = ea.Scalars(tag)
    return (
        np.array([e.step for e in evs], dtype=float),
        np.array([e.value for e in evs], dtype=float),
        np.array([e.wall_time for e in evs], dtype=float),
    )

accs = {name: load(p) for name, p in EVENTS.items()}

In [ ]:
# --- Helpers: x-axis transforms + plotting ---
LOSS_TAGS = {
    "DiffiT b192 (256x256)": "Train/Loss/train",
    "DiffiT b256 (256x256)": "Train/Loss/train",
    "DiT-XL (256x256)":      "train/loss",
    "NiT":                   "loss_denoising",
}
FID_TAGS = [
    ("DiffiT b192 (256x256)", "DiffiT b192 (256x256)", "Metrics/FID"),
    ("DiffiT b256 (256x256)", "DiffiT b256 (256x256)", "Metrics/FID"),
    ("DiT-XL (256x256)",      "DiT-XL (256x256)",      "eval/fid"),
    ("NiT (256x256)",         "NiT",                   "fid_256x256"),
    ("NiT (512x512)",         "NiT",                   "fid_512x512"),
    ("NiT (1024x1024)",       "NiT",                   "fid_1024x1024"),
]

def transform_x(model_key, steps, walls, mode):
    """Return (x_values, x_title, log_x) for the requested normalization mode."""
    if mode == "step":
        return steps, "TB step (NOT comparable across models)", False
    if mode == "samples":
        return steps * STEP_TO_IMAGES[model_key], "images seen", True
    if mode == "kimg":
        return steps * STEP_TO_IMAGES[model_key] / 1000.0, "kimg (thousands of images)", True
    if mode == "relative":
        m = steps.max() if steps.size else 1.0
        return steps / m if m > 0 else steps, "relative progress (step / max step)", False
    if mode == "wall":
        if walls.size == 0:
            return walls, "wall-clock hours from start", False
        return (walls - walls[0]) / 3600.0, "wall-clock hours from start", False
    raise ValueError(mode)

def smooth(y, k=50):
    if len(y) < k:
        return y
    c = np.cumsum(np.insert(y, 0, 0))
    return (c[k:] - c[:-k]) / k

def plot_loss(mode):
    fig = go.Figure()
    xtitle = ""
    log_x = False
    for name, tag in LOSS_TAGS.items():
        steps, vals, walls = scalar(accs[name], tag)
        x, xtitle, log_x = transform_x(name, steps, walls, mode)
        c = COLORS[name]
        fig.add_trace(go.Scatter(x=x, y=vals, mode="lines",
                                 line=dict(color=c, width=1), opacity=0.2,
                                 showlegend=False, hoverinfo="skip"))
        ys = smooth(vals)
        xs = x[len(x) - len(ys):]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", name=name,
                                 line=dict(color=c, width=2)))
    fig.update_layout(title=f"Training loss — x-axis: {xtitle}",
                      xaxis_title=xtitle, yaxis_title="loss",
                      template="plotly_white", width=950, height=500)
    if log_x:
        fig.update_xaxes(type="log")
    return fig

def plot_fid(mode):
    fig = go.Figure()
    xtitle = ""
    log_x = False
    for label, model_key, tag in FID_TAGS:
        steps, vals, walls = scalar(accs[model_key], tag)
        x, xtitle, log_x = transform_x(model_key, steps, walls, mode)
        fig.add_trace(go.Scatter(x=x, y=vals, mode="lines+markers",
                                 name=label, line=dict(color=COLORS[label])))
    fig.update_layout(title=f"FID — x-axis: {xtitle}",
                      xaxis_title=xtitle, yaxis_title="FID",
                      template="plotly_white", width=950, height=500)
    if log_x:
        fig.update_xaxes(type="log")
    return fig

In [ ]:
# --- Option 1: raw step ---
plot_loss("step").show()
plot_fid("step").show()

In [ ]:
# --- Option 2: images seen (log-x) — fairest cross-model comparison ---
# DiffiT and DiT-XL share global batch 256, so on this axis their curves
# should align directly (the 4× horizontal offset you saw before came
# from DiffiT's TB step being kimg, not optimizer steps).
plot_loss("samples").show()
plot_fid("samples").show()

In [ ]:
# --- Option 2b: kimg (canonical diffusion unit, log-x) ---
plot_loss("kimg").show()
plot_fid("kimg").show()

In [ ]:
# --- Option 3: relative progress [0, 1] ---
plot_loss("relative").show()
plot_fid("relative").show()

In [ ]:
# --- Option 4: wall-clock hours ---
plot_loss("wall").show()
plot_fid("wall").show()

In [ ]:
# --- Phase-transition view: loss vs FID (parametric, colored by step) ---
fig = go.Figure()
for label, model_key, fid_tag in FID_TAGS:
    fid_steps, fid_vals, _ = scalar(accs[model_key], fid_tag)
    loss_steps, loss_vals, _ = scalar(accs[model_key], LOSS_TAGS[model_key])
    if fid_steps.size == 0 or loss_steps.size == 0:
        continue
    order = np.argsort(loss_steps)
    loss_at_fid = np.interp(fid_steps, loss_steps[order], loss_vals[order])
    fig.add_trace(go.Scatter(
        x=loss_at_fid, y=fid_vals, mode="lines+markers", name=label,
        line=dict(color=COLORS[label]),
        marker=dict(size=7, color=fid_steps, colorscale="Viridis",
                    showscale=False, line=dict(width=0)),
        hovertemplate="loss=%{x:.4f}<br>FID=%{y:.2f}<br>step=%{marker.color:.0f}<extra>%{fullData.name}</extra>",
    ))
fig.update_layout(
    title="Phase transition: FID vs training loss (markers colored by step)",
    xaxis_title="loss (interpolated to FID step)",
    yaxis_title="FID",
    template="plotly_white", width=950, height=550,
)
fig.update_yaxes(type="log")
fig.show()